In [1]:
import pickle
from collections import OrderedDict
import os
import torch
import numpy as np
import pandas as pd
import sklearn
import random
import h5py
from sklearn.model_selection import GroupKFold
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import scanpy as sc
import anndata as ad
from textwrap import shorten

In [2]:
adata = sc.read('./Data/FinalData/adataAfterClean.h5ad')
sc.pp.normalize_total(adata)

In [3]:
## 1 Locate control and treatment rows
# Boolean masks
ctrl_mask   = adata.obs["control"] == 1      # controls
treat_mask  = ~ctrl_mask                     # treatments

# Integer indices of treated rows
treat_idx   = np.where(treat_mask)[0]        # shape (n_treat,)
n_treat     = len(treat_idx)

print(f"{n_treat:,} treated experiments found") # 836649, correct

836,649 treated experiments found


In [4]:
treat_idx

array([     0,      1,      2, ..., 836646, 836647, 836648])

In [5]:
ctrl_mask

index
REP.A001_A375_24H_X1_B22:B13-2    False
REP.A001_A375_24H_X1_B22:B14-2    False
REP.A001_A375_24H_X1_B22:B15-2    False
REP.A001_A375_24H_X1_B22:B16-2    False
REP.A001_A375_24H_X1_B22:B17-2    False
                                  ...  
PCLB003_PC3_24H_X3_B13:A04-1       True
PCLB003_PC3_24H_X3_B13:A05-1       True
PCLB003_PC3_24H_X3_B13:A06-1       True
PCLB003_PC3_24H_X3_B13:B04-1       True
PCLB003_PC3_24H_X3_B13:B05-1       True
Name: control, Length: 883077, dtype: bool

In [6]:
treat_mask

index
REP.A001_A375_24H_X1_B22:B13-2     True
REP.A001_A375_24H_X1_B22:B14-2     True
REP.A001_A375_24H_X1_B22:B15-2     True
REP.A001_A375_24H_X1_B22:B16-2     True
REP.A001_A375_24H_X1_B22:B17-2     True
                                  ...  
PCLB003_PC3_24H_X3_B13:A04-1      False
PCLB003_PC3_24H_X3_B13:A05-1      False
PCLB003_PC3_24H_X3_B13:A06-1      False
PCLB003_PC3_24H_X3_B13:B04-1      False
PCLB003_PC3_24H_X3_B13:B05-1      False
Name: control, Length: 883077, dtype: bool

In [7]:
## 2 Find each treatment’s matched control row
# 2-A  mapping from row name → integer position
name_to_row = {name: i for i, name in enumerate(adata.obs_names)}

# 2-B  string IDs of each treatment’s paired control
paired_ctrl_names = (
    adata.obs.loc[treat_mask, "paired_control_index"]
         .astype(str)
         .to_numpy()
)

# 2-C  convert those names to integer indices (needed only for slicing)
paired_ctrl_idx = np.array([name_to_row[n] for n in paired_ctrl_names], dtype=int)

# sanity check
assert paired_ctrl_idx.shape == (n_treat,), "Length mismatch"
print("First 3 control row names & indices:",
      list(zip(paired_ctrl_names[:3], paired_ctrl_idx[:3])))

First 3 control row names & indices: [('REP.A001_A375_24H_X1_B22:B03-2', 836653), ('REP.A001_A375_24H_X1_B22:J13-2', 836663), ('REP.A001_A375_24H_X1_B22:F11-2', 836661)]


In [8]:
len(paired_ctrl_idx)

836649

In [9]:
## 3 Collect metadata arrays (canonical_smiles, cid, sig)
smiles = adata.obs.loc[treat_mask, "SMILES"].to_numpy(dtype="S")
cids   = (
    adata.obs.loc[treat_mask, "cell_id"]
         .astype(str)
         .to_numpy(dtype="S")
)

# keep the original row-name strings as sigs
sigs = adata.obs_names[treat_mask].to_numpy(dtype="S")   # no integer conversion


# Inspect if you like
print(smiles[:3], cids[:3], sigs[:3], sep="\n")

[b'CCC1(CCC(=O)NC1=O)c1ccc(N)cc1' b'CCC1(CCC(=O)NC1=O)c1ccc(N)cc1'
 b'CCC1(CCC(=O)NC1=O)c1ccc(N)cc1']
[b'A375' b'A375' b'A375']
[b'REP.A001_A375_24H_X1_B22:B13-2' b'REP.A001_A375_24H_X1_B22:B14-2'
 b'REP.A001_A375_24H_X1_B22:B15-2']


In [10]:
## 4 Extract expression matrices (x1, x2)
# Ensure dense 2-D array
X_dense = adata.X
if hasattr(adata.X, "toarray"):
    X_dense = adata.X.toarray()

x2 = X_dense[treat_idx].astype(np.float32)        # treated
x1 = X_dense[paired_ctrl_idx].astype(np.float32)  # matched controls

# Double-check shapes
assert x1.shape == x2.shape == (n_treat, adata.n_vars)
print("Expression matrix shape of x1:", x1.shape)
print("Expression matrix shape of x2:", x2.shape)

Expression matrix shape of x1: (836649, 978)
Expression matrix shape of x2: (836649, 978)


In [11]:
## 5 Write everything to processed_data.h5
# Make sure we’re looking at plain Python str objects
smiles_str = smiles.astype(str)
cids_str   = cids.astype(str)
sigs_str   = sigs.astype(str)

max_len_smiles = max(len(s) for s in smiles_str)
max_len_cid    = max(len(s) for s in cids_str)
max_len_sig    = max(len(s) for s in sigs_str)

print("Max lengths →  canonical_smiles:", max_len_smiles,
      "cid:", max_len_cid,
      "sig:", max_len_sig)

Max lengths →  canonical_smiles: 458 cid: 8 sig: 46


In [12]:
## 5.1 peek before output.
def short(s, n=40):
    """Abbreviate long SMILES strings for display."""
    return shorten(s, width=n, placeholder="…")

In [13]:
peek_rows = 5

peek_df = pd.DataFrame({
    "canonical_smiles": [short(s) for s in smiles_str[:peek_rows]],
    "cid":               cids_str[:peek_rows],
    "sig":               sigs_str[:peek_rows]
})

In [14]:
print("\n=== Metadata sample ===")
display(peek_df)                       # if using Jupyter


=== Metadata sample ===


,canonical_smiles,cid,sig
0,CCC1(CCC(=O)NC1=O)c1ccc(N)cc1,A375,REP.A001_A375_24H_X1_B22:B13-2
1,CCC1(CCC(=O)NC1=O)c1ccc(N)cc1,A375,REP.A001_A375_24H_X1_B22:B14-2
2,CCC1(CCC(=O)NC1=O)c1ccc(N)cc1,A375,REP.A001_A375_24H_X1_B22:B15-2
3,CCC1(CCC(=O)NC1=O)c1ccc(N)cc1,A375,REP.A001_A375_24H_X1_B22:B16-2
4,CCC1(CCC(=O)NC1=O)c1ccc(N)cc1,A375,REP.A001_A375_24H_X1_B22:B17-2


In [15]:
print("\n=== Array shapes ===")
print("x1:", x1.shape, " x2:", x2.shape)

print("\n=== First 10 genes of first sample ===")
print("x1[0, :10] =", x1[0, :10])
print("x2[0, :10] =", x2[0, :10])


=== Array shapes ===
x1: (836649, 978)  x2: (836649, 978)

=== First 10 genes of first sample ===
x1[0, :10] = [ 7.162834   4.5501533  9.677956  10.9054985  9.161943   8.555777
 13.348677   9.774514   6.1438174 13.564107 ]
x2[0, :10] = [ 6.9927077  4.1424923  9.461794   9.984107   9.241061   8.385045
 14.390559   9.835419   6.605875  14.256779 ]


In [16]:
len(x2[0])

978

In [ ]:
'''
double check the first example correct or not
'''

In [17]:
# ------------------------------------------------------------
# 1. Identify the first treated example
# ------------------------------------------------------------
first_sig_name = sigs_str[666]          # original row label (string)
print("Row name (sig) of first example:", first_sig_name)

# integer position of that row in adata.X
row_idx = adata.obs_names.get_loc(first_sig_name)
# (equivalently: row_idx = treat_idx[0]  if you kept treat_idx)
print("Integer index in adata.X:", row_idx)

# ------------------------------------------------------------
# 2. Optional sanity check: does it match x2[0]?
# ------------------------------------------------------------
# fetch the row directly from adata.X
row_expr = adata.X[row_idx]
if hasattr(row_expr, "toarray"):      # sparse -> dense
    row_expr = row_expr.toarray()

# they should be identical (within floating-point tolerance)
same = np.allclose(row_expr.astype(np.float32), x2[666])
print("Does adata.X[row_idx] equal x2[666]? →", same)

Row name (sig) of first example: REP.A001_HA1E_24H_X1_B22:P17-2
Integer index in adata.X: 666
Does adata.X[row_idx] equal x2[666]? → True


In [18]:
ctrl_row_idx   = paired_ctrl_idx[666]
ctrl_row_name  = adata.obs_names[ctrl_row_idx]

print("Matched control row:", ctrl_row_name, "(index", ctrl_row_idx, ")")
print("Does adata.X[ctrl_row_idx] equal x1[666]? →",
      np.allclose(adata.X[ctrl_row_idx].toarray() if hasattr(adata.X, "toarray") else adata.X[ctrl_row_idx],
                  x1[666]))

Matched control row: REP.A001_HA1E_24H_X1_B22:J17-2 (index 836727 )
Does adata.X[ctrl_row_idx] equal x1[666]? → True


In [19]:
# ------------------------------------------------------------
# 3. Verify canonical_smiles and cid for the same example
# ------------------------------------------------------------
# Retrieve the obs row as a Series using the row name
obs_row = adata.obs.loc[first_sig_name]

expected_smiles = obs_row["SMILES"]
expected_cid    = str(obs_row["cell_id"])

print("\n=== Metadata cross-check for first example ===")
print(f"  stored canonical_smiles : {smiles_str[666]}")
print(f"  adata.obs['SMILES']     : {expected_smiles}")
print("  match? →", smiles_str[666] == expected_smiles)

print(f"\n  stored cid              : {cids_str[666]}")
print(f"  adata.obs['cell_id']    : {expected_cid}")
print("  match? →", cids_str[666] == expected_cid)


=== Metadata cross-check for first example ===
  stored canonical_smiles : Oc1cc2c3c(oc(=O)c4cc(O)c(O)c(oc2=O)c34)c1O
  adata.obs['SMILES']     : Oc1cc2c3c(oc(=O)c4cc(O)c(O)c(oc2=O)c34)c1O
  match? → True

  stored cid              : HA1E
  adata.obs['cell_id']    : HA1E
  match? → True


In [20]:
print(x1[666, :10])
print(x2[666, :10])

[ 6.3783407  8.157226   7.112795  10.02081    8.818842   8.960301
 13.230861   9.894741   5.811619  14.077758 ]
[ 6.329581   8.341084   7.152749   9.898995   9.051124   8.742913
 13.2307825  9.769594   5.537097  14.264521 ]


In [24]:
adata.X[row_idx][0:10]

array([ 6.329581 ,  8.341084 ,  7.152749 ,  9.898995 ,  9.051124 ,
        8.742913 , 13.2307825,  9.769594 ,  5.537097 , 14.264521 ],
      dtype=float32)

In [23]:
adata.X[ctrl_row_idx][0:10]

array([ 6.3783407,  8.157226 ,  7.112795 , 10.02081  ,  8.818842 ,
        8.960301 , 13.230861 ,  9.894741 ,  5.811619 , 14.077758 ],
      dtype=float32)

In [ ]:
'''
check done
'''

In [25]:
out_path = "./Code/other_models/TranSiGen/Meisheng_used_data/processed_data.h5"

# ------------------------------------------------------------------
# 2. encode to fixed-length **bytes** arrays
# ------------------------------------------------------------------
smiles_b = np.asarray([s.encode('utf-8') for s in smiles_str],
                      dtype=f"S{max_len_smiles}")
cids_b   = np.asarray([s.encode('utf-8') for s in cids_str],
                      dtype=f"S{max_len_cid}")
sigs_b   = np.asarray([s.encode('utf-8') for s in sigs_str],
                      dtype=f"S{max_len_sig}")

with h5py.File(out_path, "w") as f:
    f.create_dataset("canonical_smiles", data=smiles_b)   # dtype='S…'
    f.create_dataset("cid",              data=cids_b)
    f.create_dataset("sig",              data=sigs_b)
    f.create_dataset("x1",               data=x1, dtype=np.float32)
    f.create_dataset("x2",               data=x2, dtype=np.float32)

print("Done — wrote", len(smiles_b), "samples")

Done — wrote 836649 samples


In [ ]:
'''
load data to see if the dtype is right
'''

In [2]:
# 1. Load the HDF5 file
# If the file was saved with pandas (e.g., `df.to_hdf()` or `HDFStore`):
def load_from_HDF(fname):
    """Load data from a HDF5 file to a dictionary."""
    data = dict()
    with h5py.File(fname, 'r') as f:
        for key in f:
            data[key] = np.asarray(f[key])
            if isinstance(data[key][0], np.bytes_):
                data[key] = data[key].astype(str)
    return data

LINCS_data = load_from_HDF('./Code/other_models/TranSiGen/data/Meisheng_used_data/processed_data.h5')

In [3]:
LINCS_data

{'canonical_smiles': array(['CCC1(CCC(=O)NC1=O)c1ccc(N)cc1', 'CCC1(CCC(=O)NC1=O)c1ccc(N)cc1',
        'CCC1(CCC(=O)NC1=O)c1ccc(N)cc1', ...,
        'COCC1OC(=O)c2coc3c2C1(C)C1=C(C2CCC(=O)C2(C)CC1OC(C)=O)C3=O',
        'COCC1OC(=O)c2coc3c2C1(C)C1=C(C2CCC(=O)C2(C)CC1OC(C)=O)C3=O',
        'COCC1OC(=O)c2coc3c2C1(C)C1=C(C2CCC(=O)C2(C)CC1OC(C)=O)C3=O'],
       dtype='<U458'),
 'cid': array(['A375', 'A375', 'A375', ..., 'PC3', 'PC3', 'PC3'], dtype='<U8'),
 'sig': array(['REP.A001_A375_24H_X1_B22:B13-2', 'REP.A001_A375_24H_X1_B22:B14-2',
        'REP.A001_A375_24H_X1_B22:B15-2', ...,
        'PCLB003_PC3_24H_X3_B13:P22-1', 'PCLB003_PC3_24H_X3_B13:P23-1',
        'PCLB003_PC3_24H_X3_B13:P24-1'], dtype='<U46'),
 'x1': array([[ 7.162834 ,  4.5501533,  9.677956 , ...,  7.4843287,  7.3297353,
          7.0749807],
        [ 6.851262 ,  4.199051 ,  9.5467005, ...,  8.5687065,  7.0346737,
          6.883182 ],
        [ 6.9878273,  4.5501523,  9.468927 , ...,  8.354754 ,  6.7088585,
          6.7299

In [9]:
print(LINCS_data["x1"][666][0:10])
print(LINCS_data["x2"][666][0:10])
# check this results with the above cells for the 666th row of data.

[ 6.3783407  8.157226   7.112795  10.02081    8.818842   8.960301
 13.230861   9.894741   5.811619  14.077758 ]
[ 6.329581   8.341084   7.152749   9.898995   9.051124   8.742913
 13.2307825  9.769594   5.537097  14.264521 ]


In [4]:
# ------------------------------------------------------------
# 1. overview of shapes & dtypes
# ------------------------------------------------------------
for k, v in LINCS_data.items():
    print(f"{k:17}  shape={v.shape}  dtype={v.dtype}")

canonical_smiles   shape=(836649,)  dtype=<U458
cid                shape=(836649,)  dtype=<U8
sig                shape=(836649,)  dtype=<U46
x1                 shape=(836649, 978)  dtype=float32
x2                 shape=(836649, 978)  dtype=float32


In [5]:
LINCS_data_reference = load_from_HDF('./Code/other_models/TranSiGen/data/LINCS2020/data_example/processed_data.h5')

In [6]:
# ------------------------------------------------------------
# 1. overview of shapes & dtypes
# ------------------------------------------------------------
for k, v in LINCS_data_reference.items():
    print(f"{k:17}  shape={v.shape}  dtype={v.dtype}")

canonical_smiles   shape=(100,)  dtype=<U454
cid                shape=(100,)  dtype=<U8
sig                shape=(100,)  dtype=<U13
x1                 shape=(100, 978)  dtype=float32
x2                 shape=(100, 978)  dtype=float32


In [ ]:
'''
conversion for the first h5 file done.

stored in ''./Code/other_models/TranSiGen/data/Meisheng_used_data/processed_data.h5'';
time 8/5/2025 11:43:41 PM;
6.5 GB
'''